In [1]:
%pip install -q langchain langchain-community langchain-huggingface langchain-core sentence_transformers langchain-chroma pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /mimer/NOBACKUP/groups/naiss2025-23-515/yingyi/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Part 1: The dataset
## Task 1.1. Downloading and inspecting the question answering dataset

In [2]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-06-05 13:14:50--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2584787 (2.5M) [text/plain]
Saving to: ‘ori_pqal.json.3’

ori_pqal.json.3     100%[===================>]   2.46M  --.-KB/s    in 0.03s   

2026-06-05 13:14:50 (74.9 MB/s) - ‘ori_pqal.json.3’ saved [2584787/2584787]



In [5]:
import pandas as pd
tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({"abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS+[row.LONG_ANSWER]), axis=1),
             "year": tmp_data.YEAR})
questions = pd.DataFrame({"question": tmp_data.QUESTION,
             "year": tmp_data.YEAR,
             "gold_label": tmp_data.final_decision,
             "gold_context": tmp_data.LONG_ANSWER,
             "gold_document_id": documents.index})

### Sanity Check

In [6]:
print("documents shape:", documents.shape)
print("questions shape:", questions.shape)

print("\nExample question:")
print(questions.iloc[0].question)

print("\nGold label:")
print(questions.iloc[0].gold_label)

print("\nGold document id:")
print(questions.iloc[0].gold_document_id)

print("\nExample document:")
print(documents.iloc[0].abstract[:1000])

print("\nSanity check 1.1: PASSED")

documents shape: (890, 2)
questions shape: (890, 5)

Example question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Gold label:
yes

Gold document id:
21645374

Example document:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on 

# Step 2: Configure your LangChain LM
## Task 2.1. Select a language model

In [7]:
%pip install -q transformers accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /mimer/NOBACKUP/groups/naiss2025-23-515/yingyi/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [53]:
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

model = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    model_kwargs={
        "device_map": "auto",
        "torch_dtype": "auto",
    },
    pipeline_kwargs={
        "max_new_tokens": 20,
        "max_length": None,
        "do_sample": False,
        "return_full_text": False,
    },
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Sanity Check

In [55]:
prompt = "Does Sweden locate in Europe? Answer in only yes or no."

response = model.invoke(prompt)
print(response)

print("\nSanity check 2.1: PASSED")

 Yes.

Sanity check 2.1: PASSED


# Part 3: Set up the document database
## Task 3.1. Embedding mode

In [56]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import numpy as np

embedding_model_id = "BAAI/bge-small-en-v1.5"

embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_id,
    model_kwargs={
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Sanity Check

In [57]:
test_text = "Is Sweden a good country in Europe? Answer in only yes or no."

embedding = embedding_model.embed_query(test_text)

print(type(embedding))
print(len(embedding))
print(np.array(embedding).shape)
print(embedding[:5])

<class 'list'>
384
(384,)
[-0.016609685495495796, -0.03919249773025513, -0.02159954607486725, -0.029294701293110847, 0.067874476313591]


## Task 3.2. Chunking

In [58]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

chunks = text_splitter.split_documents(texts)

### Sanity Check

In [59]:
print("Number of original documents:", len(documents))
print("Number of LangChain documents before splitting:", len(texts))
print("Number of chunks after splitting:", len(chunks))

for i in range(3):
    print("=" * 80)
    print("Chunk index:", i)
    print("Metadata:", chunks[i].metadata)
    print("Length:", len(chunks[i].page_content))
    print(chunks[i].page_content[:1000])

from collections import Counter

chunk_counts = Counter([chunk.metadata["id"] for chunk in chunks])

print("Min chunks per document:", min(chunk_counts.values()))
print("Max chunks per document:", max(chunk_counts.values()))
print("Average chunks per document:", sum(chunk_counts.values()) / len(chunk_counts))

print("\nFirst 10 document chunk counts:")
for doc_id, count in list(chunk_counts.items())[:10]:
    print(doc_id, count)

print("\nSanity check 3.2: PASSED")

Number of original documents: 890
Number of LangChain documents before splitting: 1946
Number of chunks after splitting: 1946
Chunk index: 0
Metadata: {'id': 21645374}
Length: 913
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not un

## Task 3.3. Define a vector store

In [60]:
from langchain_chroma import Chroma

persist_dir = "./pubmedqa_chroma_db"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="pubmedqa_rag",
    persist_directory=persist_dir,
    collection_metadata={"hnsw:space": "cosine"}
)

### Sanity Check

In [61]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

print("\nSanity check 3.3: PASSED")

* [SIM=0.209706] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD) [{'id': 21645374}]
* [SIM=0.209706] Programmed cell death (PCD) is th

# Part 4: Implementing the system##  Task 4.1. Defining the full RAG pipelin (Option B)e

In [62]:
from operator import itemgetter
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

retriever = vector_store.as_retriever(
    search_kwargs={"k": 1}
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_template = """
You are answering a medical question using the provided context.

Use the context to answer the question.
Answer with only one word: Yes or No.

Context:
{context}

Question:
{question}

Answer:
"""

rag_prompt = PromptTemplate.from_template(rag_template)

rag_inputs = RunnableParallel(
    context=RunnableLambda(itemgetter("question")) | retriever,
    question=RunnableLambda(itemgetter("question"))
)

answer_chain = (
    {
        "context": RunnableLambda(itemgetter("context")) | RunnableLambda(format_docs),
        "question": RunnableLambda(itemgetter("question"))
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

rag_chain = rag_inputs.assign(answer=answer_chain)

### Sanity Check

In [63]:
sample_question = questions.iloc[0].question

rag_output = rag_chain.invoke({
    "question": sample_question
})

print("Question:")
print(rag_output["question"])

print("\nRetrieved document metadata:")
print(rag_output["context"][0].metadata)

print("\nRetrieved context:")
print(rag_output["context"][0].page_content[:1500])

print("\nRAG answer:")
print(rag_output["answer"])

print("\nSanity check 4.1: PASSED")

Question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Retrieved document metadata:
{'id': 21645374}

Retrieved context:
. Window stage leaves were stained with the mitochondrial dye MitoTracker Red CMXRos and examined. Mitochondrial dynamics were delineated into four categories (M1-M4) based on characteristics including distribution, motility, and membrane potential (ΔΨm). A TUNEL assay showed fragmented nDNA in a gradient over these mitochondrial stages. Chloroplasts and transvacuolar strands were also examined using live cell imaging. The possible importance of mitochondrial permeability transition pore (PTP) formation during PCD was indirectly examined via in vivo cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves with a significantly lower number of perforations compared to controls, and that displayed mitochondrial dynamics similar to that of non-PCD cells. Results depicted mitochondrial dynamics in vivo as 

# Part 5: Evaluate RAG on the dataset

## Task 5.1. High-level evaluation

In [71]:
def ask_rag(question):
    output = rag_chain.invoke({"question": question})
    return {
        "question": output["question"],
        "answer": output["answer"].strip(),
        "context": output["context"],
        "retrieved_document_id": output["context"][0].metadata["id"] if len(output["context"]) > 0 else None
    }

def clean_yes_no(answer):
    answer = answer.strip().lower()
    
    if answer.startswith("yes"):
        return "yes"
    elif answer.startswith("no"):
        return "no"
    else:
        return "invalid"

def accuracy_score_simple(y_true, y_pred):
    correct = 0
    total = 0
    
    for true, pred in zip(y_true, y_pred):
        if pred != "invalid":
            total += 1
            if true == pred:
                correct += 1
    
    if total == 0:
        return 0
    
    return correct / total

def f1_score_yes(y_true, y_pred):
    tp = 0
    fp = 0
    fn = 0
    
    for true, pred in zip(y_true, y_pred):
        if pred == "invalid":
            continue
        
        if pred == "yes" and true == "yes":
            tp += 1
        elif pred == "yes" and true == "no":
            fp += 1
        elif pred == "no" and true == "yes":
            fn += 1
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    if precision + recall == 0:
        return 0
    
    return 2 * precision * recall / (precision + recall)

In [72]:
N = 20
eval_questions = questions[questions["gold_label"].isin(["yes", "no"])].head(N)

rag_results = []

for idx, row in eval_questions.iterrows():
    output = ask_rag(row["question"])
    
    rag_results.append({
        "question_id": idx,
        "question": row["question"],
        "gold_label": row["gold_label"],
        "rag_answer_raw": output["answer"],
        "rag_prediction": clean_yes_no(output["answer"])
    })

rag_df = pd.DataFrame(rag_results)

rag_accuracy = accuracy_score_simple(
    rag_df["gold_label"],
    rag_df["rag_prediction"]
)

rag_f1 = f1_score_yes(
    rag_df["gold_label"],
    rag_df["rag_prediction"]
)

baseline_template = """
Answer the medical yes/no question.
Answer with only one word: Yes or No.

Question:
{question}

Answer:
"""

baseline_prompt = PromptTemplate.from_template(baseline_template)

baseline_chain = (
    baseline_prompt
    | model
    | StrOutputParser()
)

def ask_baseline(question):
    answer = baseline_chain.invoke({"question": question})
    return answer.strip()

baseline_results = []

for idx, row in eval_questions.iterrows():
    answer = ask_baseline(row["question"])
    
    baseline_results.append({
        "question_id": idx,
        "baseline_answer_raw": answer,
        "baseline_prediction": clean_yes_no(answer)
    })

baseline_df = pd.DataFrame(baseline_results)

combined_df = rag_df.merge(baseline_df, on="question_id")

baseline_accuracy = accuracy_score_simple(
    combined_df["gold_label"],
    combined_df["baseline_prediction"]
)

baseline_f1 = f1_score_yes(
    combined_df["gold_label"],
    combined_df["baseline_prediction"]
)

print("RAG accuracy:", rag_accuracy)
print("RAG F1:", rag_f1)
print("Baseline accuracy:", baseline_accuracy)
print("Baseline F1:", baseline_f1)

combined_df.head(10)

RAG accuracy: 0.7
RAG F1: 0.7857142857142857
Baseline accuracy: 0.6
Baseline F1: 0.7142857142857143


,question_id,question,gold_label,rag_answer_raw,rag_prediction,baseline_answer_raw,baseline_prediction
0,21645374,Do mitochondria play a role in remodelling lac...,yes,Yes,yes,No.,no
1,16418930,Landolt C and snellen e acuity: differences in...,no,Yes,yes,Yes\n\nThe Landolt C and Snellen E acuities ar...,yes
2,9488747,"Syncope during bathing in infants, a pediatric...",yes,Yes,yes,"Yes\n\nThis condition is known as ""infantile s...",yes
3,17208539,Are the long-term results of the transanal pul...,no,No,no,No\n\nThis is a common surgical procedure for ...,no
4,10808977,Can tailored interventions increase mammograph...,yes,Yes,yes,Yes\n\nThis answer is based on the assumption ...,yes
5,23831910,Double balloon enteroscopy: is it efficacious ...,yes,No,no,Yes\n\nExplanation: Double balloon enteroscopy...,yes
6,26852225,Is adjustment for reporting heterogeneity nece...,no,Yes,yes,No. \n\nExplanation: Adjustment for reporting ...,no
7,17113061,Do mutations causing low HDL-C promote increas...,no,No\n\nThe context indicates that while genetic...,no,Yes\n\nExplanation: Mutations that lead to low...,yes
8,10966337,A short stay or 23-hour ward in a general and ...,yes,Yes,yes,No.,no
9,25432938,Did Chile's traffic law reform push police enf...,yes,Yes,yes,No. \n\nExplanation: The question is asking if...,no


## Task 5.2. Detailed inspection

In [73]:
rag_results_52 = []

for idx, row in eval_questions.iterrows():
    output = ask_rag(row["question"])
    
    rag_results_52.append({
        "question_id": idx,
        "question": row["question"],
        "gold_label": row["gold_label"],
        "gold_document_id": row["gold_document_id"],
        "retrieved_document_id": output["retrieved_document_id"],
        "rag_answer_raw": output["answer"],
        "rag_prediction": clean_yes_no(output["answer"])
    })

rag_52_df = pd.DataFrame(rag_results_52)

rag_52_df["retrieved_correct"] = (
    rag_52_df["retrieved_document_id"].astype(str)
    == rag_52_df["gold_document_id"].astype(str)
)

retrieval_accuracy = rag_52_df["retrieved_correct"].mean()

print("Retrieval accuracy:", retrieval_accuracy)
print("Correct retrieved documents:", rag_52_df["retrieved_correct"].sum())
print("Total questions:", len(rag_52_df))

rag_52_df.head(10)

Retrieval accuracy: 0.95
Correct retrieved documents: 19
Total questions: 20


,question_id,question,gold_label,gold_document_id,retrieved_document_id,rag_answer_raw,rag_prediction,retrieved_correct
0,21645374,Do mitochondria play a role in remodelling lac...,yes,21645374,21645374,Yes,yes,True
1,16418930,Landolt C and snellen e acuity: differences in...,no,16418930,16418930,Yes,yes,True
2,9488747,"Syncope during bathing in infants, a pediatric...",yes,9488747,9488747,Yes,yes,True
3,17208539,Are the long-term results of the transanal pul...,no,17208539,17208539,No,no,True
4,10808977,Can tailored interventions increase mammograph...,yes,10808977,10808977,Yes,yes,True
5,23831910,Double balloon enteroscopy: is it efficacious ...,yes,23831910,17593459,No,no,False
6,26852225,Is adjustment for reporting heterogeneity nece...,no,26852225,26852225,Yes,yes,True
7,17113061,Do mutations causing low HDL-C promote increas...,no,17113061,17113061,No\n\nThe context indicates that while genetic...,no,True
8,10966337,A short stay or 23-hour ward in a general and ...,yes,10966337,10966337,Yes,yes,True
9,25432938,Did Chile's traffic law reform push police enf...,yes,25432938,25432938,Yes,yes,True
